In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver # 🧠 New import for memory
from langchain_core.messages import HumanMessage

# 1. Setup
load_dotenv(".env")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# 2. Vector DB (RAG)
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text_data)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = FAISS.from_texts(chunks, embeddings)
retriever = vectorstore.as_retriever()

# 3. RAG Tool
@tool
def ask_docs(question: str) -> str:
    """Answers questions about LangChain using local documentation."""
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    return f"Context from docs:\n{context}"

# 4. Agent with Persistent Memory
# The 'checkpointer' allows the agent to save its state between calls
memory = MemorySaver()
tools = [ask_docs]

# Replaces initialize_agent with the modern ReAct graph
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

# 5. Running the Chat
# Use a 'thread_id' to identify this specific conversation
config = {"configurable": {"thread_id": "user_session_001"}}

def chat(query: str):
    print(f"\n👤 User: {query}")
    # The agent now looks at the thread_id to pull up past messages
    result = agent_executor.invoke(
        {"messages": [HumanMessage(content=query)]}, 
        config=config
    )
    print(f"🤖 Gemini: {result['messages'][-1].content}")

# Test the memory:
chat("What is LangChain?")
chat("Who created it?") # Gemini will know 'it' means LangChain!

C:\Users\User\AppData\Local\Temp\ipykernel_13040\2391861250.py:39: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, checkpointer=memory)



👤 User: What is LangChain?
🤖 Gemini: 

👤 User: Who created it?
🤖 Gemini: Harrison Chase.


In [3]:
# Instead of agent_executor.invoke(...), use .stream() to see the steps
config = {"configurable": {"thread_id": "session_1"}}
input_message = {"messages": [HumanMessage(content="What is LangChain?")]}

print("--- Starting Agent Cycle ---")
for chunk in agent_executor.stream(input_message, config, stream_mode="values"):
    # This chunks list contains all messages in the state so far
    last_msg = chunk["messages"][-1]
    
    # Check if it's the LLM "Thinking" (calling a tool)
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        print(f"\n🤔 Thought: I need to use a tool.")
        print(f"🛠️ Action: {last_msg.tool_calls[0]['name']}")
        print(f"📥 Action Input: {last_msg.tool_calls[0]['args']}")
    
    # Check if it's the Tool's result
    elif last_msg.type == "tool":
        print(f"👁️ Observation: {last_msg.content}")

# Print the final answer at the very end
print(f"\n🤖 Final Answer: {last_msg.content}")

--- Starting Agent Cycle ---

🤔 Thought: I need to use a tool.
🛠️ Action: ask_docs
📥 Action Input: {'question': 'What is LangChain?'}
👁️ Observation: Context from docs:
LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

🤖 Final Answer: 
